# v5 ResUNet detector — 3D heatmap regression (base24 + BatchNorm + residual + stronger 3D aug)
Single-notebook detector experiment building on v1 (LB **0.844**, the current best). Same training
machinery / cosine LR / checkpoint-resume / eval as `v1_unet_train.ipynb`; the ONLY changes are the
detector internals: wider (`BASE 16→24`), **BatchNorm** (v1=InstanceNorm), a **residual** skip in every
ConvBlock, and **actually-effective 3D augmentation** (v1's intensity aug canceled under percentile-norm).

**This bundles 4 changes** (arch width, norm, residual, aug) — fine for a detector experiment scored
locally, but each is a `NORM`/`RESIDUAL`/`BASE`/aug flag so you can **ablate to isolate what helped**
before spending an LB slot. Downstream (v2 detect+linking) is identical, so an LB submit of these weights
is still a clean single-variable **detector swap vs v1's 0.844**.

**Setup:** GPU (T4) **ON**, Internet **ON**, competition data as input. Run All. v1 weights are NOT loadable
here (residual + BatchNorm + base24 change the conv shapes) — this trains from scratch (or resumes its OWN
`unet_res_latest.pt`). Compare the eval cell's edge-Jaccard to **v1's NN 0.808**.


In [ ]:
# --- deps (internet ON during training) ---
import subprocess, sys
try:
    import zarr; assert int(zarr.__version__.split('.')[0]) >= 3
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'zarr>=3']); import zarr

import os, glob, time
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler
import scipy.ndimage as ndi
from scipy.ndimage import binary_dilation
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max
print('zarr', zarr.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# ================= CONFIG =================
INPUT     = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TRAIN_DIR = os.path.join(INPUT, 'train')
VOXEL     = np.array([1.625, 0.40625, 0.40625])   # (z,y,x) um/voxel (anisotropic)

# target / detection
PATCH      = (64, 128, 128)   # divisible by z:4, xy:16
SIGMA      = (1.0, 3.0, 3.0)  # anisotropic Gaussian (voxels)
NORM_PCT   = (50.0, 99.8)
FG_THR     = 0.15             # bright-but-unlabeled -> ignore in loss
POS_WEIGHT = 10.0
POS_FRAC   = 0.7             # fraction of patches centered on a GT point
MATCH_UM   = 7.0             # scorer node-match radius (official metric)
GATE_UM    = 8.0            # linking gate

# ---- v5 ResUNet architecture (single-notebook detector experiment vs v1's 0.844) ----
# Ablate these to isolate what helps (v1 baseline = BASE 16, InstanceNorm, NO residual):
NORM       = 'instance'   # 'batch' | 'instance' | 'group'. RUN2 instance ALSO stalled (~0.00162, =BN's 0.00167) -> BN@batch2 DISPROVED. RUN3 isolates aug (photometric aug OFF below), instance/residual/base24 held
RESIDUAL   = True         # add a residual skip to every ConvBlock (v1=False)
# model / training
BASE       = 24           # channel width (v1=16). Try 32 if T4 memory allows (may OOM -> lower BATCH or PATCH).
# stronger 3D augmentation (v1's `v*=uniform(0.9,1.1)` CANCELED under percentile-norm -> was a no-op;
# these run AFTER normalization on the [0,1] volume so they actually change the input):
AUG_GAMMA     = (1.0, 1.0)
AUG_CONTRAST  = (1.0, 1.0)
AUG_BRIGHT    = 0.0
AUG_NOISE_STD = 0.0
BATCH      = 2
LR         = 2e-4
EPOCHS     = 60            # convergence run: v1 stopped at 30 still trending down -> continue to 60
STEPS      = 300            # steps per epoch
N_VAL      = 20            # last N samples held out
NUM_WORKERS= 2
HM_THR     = 0.3           # heatmap peak threshold at inference (density / T_true knob)

# LR schedule: cosine anneal over the full EPOCHS horizon (v1 used a FLAT lr -> a big reason it
# never converged). base LR -> ETA_MIN by epoch EPOCHS. Resuming from the flat-lr epoch-29 ckpt,
# the scheduler is fast-forwarded to start_epoch, so lr continues from ~mid-cosine and decays down.
COSINE_LR  = True
ETA_MIN    = LR * 0.02     # floor lr (= 4e-6)

# ---- EVAL-ONLY mode: skip training, load uploaded weights, just score val edge-Jaccard ----
EVAL_ONLY    = False  # True = don't train; load weights from an input dataset and score. False = normal training.
EVAL_WEIGHTS = None   # explicit path to a .pt under /kaggle/input; None = auto-find (v5_UNet_res_best.pt / unet_res_latest.pt)
EVAL_N_VAL   = 5      # held-out val samples to score in the eval cell (raise toward N_VAL=20 for a firmer number)

# early stopping + time-budget safe-exit + resume
EARLY_STOP_PATIENCE = 8            # stop if val loss doesn't improve for this many epochs (raised for the decay tail)
TIME_BUDGET_SEC     = 8.5 * 3600   # stop & checkpoint before Kaggle's ~9h session limit
RESERVE_SEC         = 900          # time reserved for final eval + checkpoint saving
NB_START            = time.time()  # notebook start (budget measured from here)

# checkpoints (in /kaggle/working). To RESUME: add these as an input dataset next run.
OUT_WEIGHTS = '/kaggle/working/v5_UNet_res_best.pt'  # BEST model weights (state_dict) for eval/inference
CKPT_LATEST = '/kaggle/working/unet_res_latest.pt'   # FULL checkpoint (model+opt+scaler+sched+epoch+best+history)
HISTORY_CSV = '/kaggle/working/history.csv'       # per-epoch training trend (epoch,train,val,best,lr,sec)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# ================= IO =================
def open_image(zarr_path):
    g = zarr.open(zarr_path, mode='r')
    if hasattr(g, 'shape'):
        return g
    best, bestsize = None, -1
    def walk(node):
        nonlocal best, bestsize
        try:
            for k in node.keys():
                sub = node[k]
                if hasattr(sub, 'shape'):
                    s = int(np.prod(sub.shape))
                    if s > bestsize: best, bestsize = sub, s
                else: walk(sub)
        except Exception: pass
    walk(g); return best

def load_geff(geff_path):
    g = zarr.open(geff_path, mode='r')
    node_ids = np.asarray(g['nodes/ids'][:])
    props = {k: np.asarray(g[f'nodes/props/{k}/values'][:]) for k in g['nodes/props'].keys()}
    try: edges = np.asarray(g['edges/ids'][:])
    except Exception: edges = np.zeros((0, 2), dtype=np.uint64)
    return node_ids, props, edges

def points_by_frame(geff_path):
    _, props, _ = load_geff(geff_path)
    frames = defaultdict(list)
    for i in range(len(props['t'])):
        frames[int(props['t'][i])].append((int(props['z'][i]), int(props['y'][i]), int(props['x'][i])))
    return dict(frames)

def gt_nodes_edges(geff_path):
    node_ids, props, edges = load_geff(geff_path)
    nodes = [(int(node_ids[i]), int(props['t'][i]), int(props['z'][i]), int(props['y'][i]), int(props['x'][i]))
             for i in range(len(node_ids))]
    return nodes, [(int(s), int(t)) for s, t in edges]

def build_sample_list(train_dir):
    out = []
    for gp in sorted(glob.glob(os.path.join(train_dir, '*.geff'))):
        zp = gp[:-5] + '.zarr'
        if os.path.exists(zp): out.append((zp, gp))
    return out

In [ ]:
# ================= TARGETS: Gaussian heatmap + ignore mask =================
def _normalize(vol, norm_pct=NORM_PCT):
    v = vol.astype(np.float32)
    lo, hi = np.percentile(v, norm_pct)
    # np.percentile returns float64 -> cast back to float32 (else model gets a DoubleTensor)
    return np.clip((v - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)

def _stamp_gaussian(hm, center, sigma, radius):
    shape = hm.shape; slices, locals_ = [], []
    for i in range(3):
        ci = int(round(center[i]))
        lo = max(0, ci - radius[i]); hi = min(shape[i], ci + radius[i] + 1)
        if lo >= hi: return
        slices.append(slice(lo, hi)); locals_.append(np.arange(lo, hi) - ci)
    zz, yy, xx = np.meshgrid(*locals_, indexing='ij')
    g = np.exp(-(zz**2/(2*sigma[0]**2) + yy**2/(2*sigma[1]**2) + xx**2/(2*sigma[2]**2)))
    sl = tuple(slices); hm[sl] = np.maximum(hm[sl], g.astype(np.float32))

def make_targets(vol, points_zyx, sigma=SIGMA, norm_pct=NORM_PCT, fg_thr=FG_THR,
                 pos_weight=POS_WEIGHT, near_eps=0.05, dilate_iter=2):
    """Return (heatmap, weight). weight=0 in ignore regions (bright-but-unlabeled)."""
    shape = vol.shape
    radius = tuple(max(1, int(round(3*s))) for s in sigma)
    hm = np.zeros(shape, np.float32)
    for p in points_zyx: _stamp_gaussian(hm, p, sigma, radius)
    v = _normalize(vol, norm_pct)
    near = hm > near_eps
    near_dil = binary_dilation(near, iterations=dilate_iter) if near.any() else near
    ignore = (v > fg_thr) & ~near_dil
    weight = np.ones(shape, np.float32)
    weight[ignore] = 0.0
    weight[near] = pos_weight
    return hm, weight

In [ ]:
# ================= DATASET =================
def _rand_center(point, patch, shape, jitter=0.25):
    c = []
    for i in range(3):
        half = patch[i] // 2
        if point is not None:
            ci = int(point[i]) + int(np.random.uniform(-jitter, jitter) * patch[i])
        else:
            ci = np.random.randint(0, shape[i])
        ci = min(max(ci, half), max(half, shape[i] - (patch[i] - half)))
        c.append(ci)
    return c

def _crop(arr, center, patch):
    sl = tuple(slice(center[i]-patch[i]//2, center[i]-patch[i]//2+patch[i]) for i in range(3))
    return arr[sl]

class HeatmapDataset(Dataset):
    def __init__(self, samples, patch=PATCH, sigma=SIGMA, norm_pct=NORM_PCT, fg_thr=FG_THR,
                 pos_weight=POS_WEIGHT, pos_frac=POS_FRAC, augment=True):
        self.samples, self.patch, self.sigma = samples, patch, sigma
        self.norm_pct, self.fg_thr, self.pos_weight = norm_pct, fg_thr, pos_weight
        self.pos_frac, self.augment = pos_frac, augment
        self._cache = {}
        self.items = []
        for si, (_, gp) in enumerate(samples):
            for t, pts in points_by_frame(gp).items():
                self.items.append((si, t, pts))

    def _arr(self, si):
        if si not in self._cache: self._cache[si] = open_image(self.samples[si][0])
        return self._cache[si]

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        si, t, pts = self.items[idx]
        vol = np.asarray(self._arr(si)[t]).astype(np.float32)
        hm, w = make_targets(vol, pts, self.sigma, self.norm_pct, self.fg_thr, self.pos_weight)
        point = pts[np.random.randint(len(pts))] if (pts and np.random.rand() < self.pos_frac) else None
        c = _rand_center(point, self.patch, vol.shape)
        v, h, m = _crop(vol, c, self.patch), _crop(hm, c, self.patch), _crop(w, c, self.patch)
        if self.augment:
            for ax in range(3):                                    # flips (z,y,x) -- label-preserving
                if np.random.rand() < 0.5:
                    v, h, m = np.flip(v, ax), np.flip(h, ax), np.flip(m, ax)
            k = np.random.randint(4)                               # 0/90/180/270 in-plane (xy patch is square)
            if k:
                v = np.rot90(v, k, (1, 2)); h = np.rot90(h, k, (1, 2)); m = np.rot90(m, k, (1, 2))
        lo, hi = np.percentile(v, self.norm_pct)
        v = np.clip((v - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)
        if self.augment:                                           # intensity augs on the NORMALIZED [0,1] vol
            if np.random.rand() < 0.5:                             #   (v1 scaled raw v then re-normed -> canceled)
                v = np.clip(v ** np.float32(np.random.uniform(*AUG_GAMMA)), 0, 1)
            if np.random.rand() < 0.5:
                v = np.clip(v * np.float32(np.random.uniform(*AUG_CONTRAST))
                            + np.float32(np.random.uniform(-AUG_BRIGHT, AUG_BRIGHT)), 0, 1).astype(np.float32)
            if AUG_NOISE_STD > 0 and np.random.rand() < 0.5:
                v = np.clip(v + np.random.normal(0, AUG_NOISE_STD, v.shape).astype(np.float32), 0, 1)
        to_t = lambda a: torch.from_numpy(np.ascontiguousarray(a)[None])
        return to_t(v), to_t(h), to_t(m)

In [ ]:
# ================= MODEL: anisotropic 3D U-Net =================
STRIDES = ((1,2,2), (1,2,2), (2,2,2), (2,2,2))   # downsample xy first, z later

def make_norm(norm, ch):
    if norm == 'batch':    return nn.BatchNorm3d(ch)
    if norm == 'instance': return nn.InstanceNorm3d(ch, affine=True)
    if norm == 'group':    return nn.GroupNorm(min(8, ch), ch)   # small-batch-safe fallback
    raise ValueError(f'unknown NORM {norm!r}')

class ConvBlock(nn.Module):
    """Two 3x3x3 convs (norm = NORM). Optional RESIDUAL skip: out = act(convs(x) + proj(x)),
    proj = 1x1 conv only when channels change. RESIDUAL=False + NORM='instance' == v1's block."""
    def __init__(self, cin, cout, norm=None, residual=None):
        super().__init__()
        norm = NORM if norm is None else norm
        self.residual = RESIDUAL if residual is None else residual
        self.conv1 = nn.Conv3d(cin, cout, 3, padding=1, bias=False)
        self.n1    = make_norm(norm, cout)
        self.conv2 = nn.Conv3d(cout, cout, 3, padding=1, bias=False)
        self.n2    = make_norm(norm, cout)
        self.act   = nn.LeakyReLU(0.01, inplace=True)
        self.proj  = nn.Conv3d(cin, cout, 1, bias=False) if (self.residual and cin != cout) else None
    def forward(self, x):
        out = self.act(self.n1(self.conv1(x)))
        out = self.n2(self.conv2(out))
        if self.residual:
            out = out + (self.proj(x) if self.proj is not None else x)
        return self.act(out)

class UNet3D(nn.Module):
    def __init__(self, in_ch=1, base=BASE, strides=STRIDES):
        super().__init__()
        n = len(strides); chs = [base*(2**i) for i in range(n+1)]
        self.enc, self.downs = nn.ModuleList(), nn.ModuleList()
        cin = in_ch
        for i in range(n):
            self.enc.append(ConvBlock(cin, chs[i]))
            self.downs.append(nn.Conv3d(chs[i], chs[i], kernel_size=strides[i], stride=strides[i]))
            cin = chs[i]
        self.bottleneck = ConvBlock(chs[n-1], chs[n])
        self.ups, self.dec = nn.ModuleList(), nn.ModuleList()
        for i in reversed(range(n)):
            self.ups.append(nn.ConvTranspose3d(chs[i+1], chs[i], kernel_size=strides[i], stride=strides[i]))
            self.dec.append(ConvBlock(chs[i]*2, chs[i]))
        self.head = nn.Conv3d(chs[0], 1, 1)
    def forward(self, x):
        skips = []
        for enc, down in zip(self.enc, self.downs):
            x = enc(x); skips.append(x); x = down(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.ups, self.dec, reversed(skips)):
            x = up(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.head(x)   # raw logits; sigmoid in loss/inference

In [ ]:
# ================= LOSS / LINK / SCORE / INFER =================
def masked_mse(logits, target, weight):
    pred = torch.sigmoid(logits)
    se = (pred - target)**2 * weight
    return se.sum() / (weight.sum() + 1e-6)

def build_graph(coords_list, gate=GATE_UM):
    nodes, edges, nid = [], [], 0
    prev_ids, prev_xyz = None, None
    for t, coords in enumerate(coords_list):
        ids = list(range(nid, nid+len(coords))); nid += len(coords)
        for i, c in zip(ids, coords): nodes.append((i, t, int(c[0]), int(c[1]), int(c[2])))
        if prev_xyz is not None and len(prev_xyz) and len(coords):
            D = cdist(prev_xyz*VOXEL, coords*VOXEL)
            r, c = linear_sum_assignment(D)
            for i, j in zip(r, c):
                if D[i, j] <= gate: edges.append((prev_ids[i], ids[j]))
        prev_ids, prev_xyz = ids, coords
    return nodes, edges

def match_nodes_per_t(gt_nodes, pred_nodes, max_um=MATCH_UM):
    gt_by_t, pr_by_t = defaultdict(list), defaultdict(list)
    for n in gt_nodes: gt_by_t[n[1]].append(n)
    for n in pred_nodes: pr_by_t[n[1]].append(n)
    g2p = {}
    for t, G in gt_by_t.items():
        P = pr_by_t.get(t, [])
        if not P: continue
        A = np.array([[g[2],g[3],g[4]] for g in G])*VOXEL
        B = np.array([[p[2],p[3],p[4]] for p in P])*VOXEL
        D = cdist(A, B); r, c = linear_sum_assignment(D)
        for i, j in zip(r, c):
            if D[i, j] <= max_um: g2p[G[i][0]] = P[j][0]
    return g2p

def edge_jaccard(gt_nodes, gt_edges, pred_nodes, pred_edges, max_um=MATCH_UM):
    g2p = match_nodes_per_t(gt_nodes, pred_nodes, max_um)
    p2g = {v: k for k, v in g2p.items()}
    pred_set, gt_set = set(map(tuple, pred_edges)), set(map(tuple, gt_edges))
    TP = FP = 0
    for a, b in pred_edges:
        if a in p2g and b in p2g:
            TP += (p2g[a], p2g[b]) in gt_set; FP += (p2g[a], p2g[b]) not in gt_set
    FN = sum(1 for u, v in gt_edges if not (u in g2p and v in g2p and (g2p[u], g2p[v]) in pred_set))
    d = TP+FP+FN
    return dict(jaccard=TP/d if d else 0.0, TP=TP, FP=FP, FN=FN,
                matched_gt=len(g2p), n_gt_nodes=len(gt_nodes), n_pred_nodes=len(pred_nodes))

@torch.no_grad()
def predict_heatmap(model, vol, norm_pct=NORM_PCT):
    v = _normalize(vol, norm_pct)
    x = torch.from_numpy(np.ascontiguousarray(v)[None, None]).float().to(device)  # ensure float32
    with torch.amp.autocast(device, enabled=(device=='cuda')):
        y = torch.sigmoid(model(x))
    return y[0, 0].float().cpu().numpy()

def detect_all_unet(model, arr, threshold=HM_THR, min_distance=3):
    model.eval(); out = []
    for t in range(arr.shape[0]):
        hm = predict_heatmap(model, np.asarray(arr[t]))
        out.append(peak_local_max(hm, min_distance=min_distance, threshold_abs=threshold,
                                  exclude_border=False, num_peaks=5000))
    return out

In [ ]:
# ================= build datasets =================
samples = build_sample_list(TRAIN_DIR)
train_s, val_s = samples[:-N_VAL], samples[-N_VAL:]
print(f'{len(samples)} samples -> train {len(train_s)}, val {len(val_s)}')

train_ds = HeatmapDataset(train_s, augment=True)
val_ds   = HeatmapDataset(val_s, augment=False)
print('train frames:', len(train_ds), '| val frames:', len(val_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH,
                          sampler=RandomSampler(train_ds, replacement=True, num_samples=STEPS*BATCH),
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

v, h, m = next(iter(train_loader))
print('batch:', v.shape, '| hm max', float(h.max()), '| weight>0 frac', float((m>0).float().mean()))

In [ ]:
# ================= TRAIN (AMP + resume/warm-start + cosine-LR + early-stop + time-budget safe-exit) =================
# EVAL_ONLY short-circuit: load uploaded weights and SKIP training; the next cell scores val edge-Jaccard.
import csv

model = UNet3D().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler(device, enabled=(device=='cuda'))
print('params (M):', sum(p.numel() for p in model.parameters())/1e6)

@torch.no_grad()
def val_loss():
    model.eval(); tot = n = 0
    for v, h, m in val_loader:
        v, h, m = v.to(device), h.to(device), m.to(device)
        with torch.amp.autocast(device, enabled=(device=='cuda')):
            tot += float(masked_mse(model(v), h, m))
        n += 1
        if n >= 60: break
    return tot / max(n, 1)

def write_history_csv(history):
    with open(HISTORY_CSV, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['epoch', 'train', 'val', 'best', 'lr', 'sec'])
        w.writeheader(); w.writerows(history)

def save_ckpt(epoch, best, patience, history, best_state, sched):
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                    sched=(sched.state_dict() if sched is not None else None),
                    epoch=epoch, best=best, patience=patience, history=history,
                    best_model=best_state), CKPT_LATEST)
    write_history_csv(history)

def _find_eval_weights():
    """Locate an uploaded .pt (best-weights state_dict OR full checkpoint) under /kaggle/input."""
    if EVAL_WEIGHTS:
        return EVAL_WEIGHTS
    for pat in ('v5_UNet_res_best.pt', 'unet_res_latest.pt', '*.pt'):
        hits = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
        if hits:
            return hits[0]
    return None

def _extract_state(ck):
    """Return a plain state_dict whether ck is a state_dict or a full checkpoint dict."""
    if isinstance(ck, dict) and ('best_model' in ck or 'model' in ck):
        return ck.get('best_model') or ck['model']
    return ck

if EVAL_ONLY:
    wpath = _find_eval_weights()
    assert wpath, 'EVAL_ONLY: no .pt found under /kaggle/input -- add your weights dataset as an input.'
    state = _extract_state(torch.load(wpath, map_location=device))
    model.load_state_dict(state); model.eval()
    torch.save(state, OUT_WEIGHTS)   # expose at OUT_WEIGHTS so the eval cell loads it unchanged
    print(f'EVAL_ONLY: loaded weights from {wpath} -> training SKIPPED. Run the next cell for edge-Jaccard.')
else:
    # ---- resume / warm-start from an uploaded dataset (add it as an input next run) ----
    #   * unet_latest.pt (FULL ckpt: model+opt+scaler+sched+epoch...) -> exact resume, epoch continues.
    #   * else v1_UNet_best.pt / unet_heatmap.pt (best WEIGHTS only)  -> warm-start: fresh opt, continue
    #     from the cosine midpoint so LR is gentle (~1e-4), not a 2e-4 restart that could damage the weights.
    start_epoch, best, patience, history, best_state, sched_state = 0, 1e9, 0, [], None, None
    resume_hits = glob.glob('/kaggle/input/**/unet_res_latest.pt', recursive=True)
    if resume_hits:
        ck = torch.load(resume_hits[0], map_location=device)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); scaler.load_state_dict(ck['scaler'])
        start_epoch = ck['epoch'] + 1; best = ck['best']; patience = ck.get('patience', 0)
        history = ck.get('history', []); best_state = ck.get('best_model', None); sched_state = ck.get('sched', None)
        if best_state is not None: torch.save(best_state, OUT_WEIGHTS)  # restore best weights file
        print(f'RESUMED (full ckpt) from {resume_hits[0]} -> start epoch {start_epoch}, best val {best:.6f}, '
              f'patience {patience}' + (' (no sched in ckpt -> fast-forwarding cosine)'
                                        if (COSINE_LR and sched_state is None) else ''))
    else:
        warm = None
        for pat in ('v5_UNet_res_best.pt',):   # v5-only: v1/v4 weights have different conv shapes, NOT loadable
            hits = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
            if hits: warm = hits[0]; break
        if warm:
            state = _extract_state(torch.load(warm, map_location=device))
            model.load_state_dict(state)
            best_state = {k: v.detach().cpu() for k, v in state.items()}
            torch.save(state, OUT_WEIGHTS)
            best = val_loss()                 # seed best from the warm model -> never end worse than we started
            start_epoch = EPOCHS // 2         # continue as if mid-run so the cosine LR resumes gently (~1e-4)
            print(f'WARM-START from best weights {warm} -> fresh opt, continue from epoch {start_epoch}/{EPOCHS} '
                  f'(gentle cosine LR), seeded best val {best:.6f}.')
        else:
            print('fresh start (no checkpoint or weights found under /kaggle/input)')

    # ---- cosine LR schedule over the full EPOCHS horizon; resume/warm-start safe ----
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=ETA_MIN) if COSINE_LR else None
    if sched is not None:
        if sched_state is not None:
            sched.load_state_dict(sched_state)                 # exact resume from a new-style ckpt
        elif start_epoch > 0:
            for _ in range(start_epoch): sched.step()          # fast-forward past prior/flat-lr epochs
        print(f'cosine LR: T_max={EPOCHS}, eta_min={ETA_MIN:.2e}, lr now {opt.param_groups[0]["lr"]:.2e}')

    epoch_times = []
    for ep in range(start_epoch, EPOCHS):
        # time-budget guard: if another epoch (+ reserve for eval/save) risks the session limit, stop safely
        if epoch_times:
            avg = sum(epoch_times) / len(epoch_times)
            elapsed = time.time() - NB_START
            if elapsed + avg + RESERVE_SEC > TIME_BUDGET_SEC:
                print(f'TIME BUDGET: pausing before epoch {ep} (elapsed {elapsed/3600:.2f}h, ~{avg:.0f}s/epoch). '
                      f'Checkpoint saved -> upload unet_latest.pt next run to resume.')
                break
        cur_lr = opt.param_groups[0]['lr']
        model.train(); t0 = time.time(); run = 0.0
        for i, (v, h, m) in enumerate(train_loader):
            v, h, m = v.to(device), h.to(device), m.to(device)
            with torch.amp.autocast(device, enabled=(device=='cuda')):
                loss = masked_mse(model(v), h, m)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += float(loss)
        vl = val_loss(); dt = time.time() - t0; epoch_times.append(dt)
        if vl < best:
            best = vl; patience = 0
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            torch.save(best_state, OUT_WEIGHTS)
        else:
            patience += 1
        if sched is not None: sched.step()   # advance cosine BEFORE checkpoint so resume lr is exact
        history.append(dict(epoch=ep, train=round(run/(i+1), 6), val=round(vl, 6), best=round(best, 6),
                            lr=round(cur_lr, 8), sec=round(dt)))
        save_ckpt(ep, best, patience, history, best_state, sched)
        rem = (TIME_BUDGET_SEC - (time.time() - NB_START)) / 3600
        print(f'epoch {ep:2d} | lr {cur_lr:.2e} | train {run/(i+1):.4f} | val {vl:.4f} | best {best:.4f} '
              f'| patience {patience}/{EARLY_STOP_PATIENCE} | {dt:.0f}s | ~{rem:.2f}h budget left')
        if patience >= EARLY_STOP_PATIENCE:
            print(f'EARLY STOP at epoch {ep} (no val improvement for {EARLY_STOP_PATIENCE} epochs).')
            break

    if best_state is not None:  # guarantee best weights file exists for the eval/inference cell
        torch.save(best_state, OUT_WEIGHTS)
    write_history_csv(history)
    print(f'done. best val {best:.6f} | weights -> {OUT_WEIGHTS} | resume ckpt -> {CKPT_LATEST} | trend -> {HISTORY_CSV}')


In [ ]:
# ================= Sanity: edge-Jaccard on VAL samples vs classical ~0.75 =================
model.load_state_dict(torch.load(OUT_WEIGHTS, map_location=device)); model.eval()
TP = FP = FN = 0
for zp, gp in val_s[:EVAL_N_VAL]:
    arr = open_image(zp)
    coords = detect_all_unet(model, arr, threshold=HM_THR, min_distance=3)
    p_nodes, p_edges = build_graph(coords, gate=GATE_UM)
    gt_nodes, gt_edges = gt_nodes_edges(gp)
    r = edge_jaccard(gt_nodes, gt_edges, p_nodes, p_edges)
    TP += r['TP']; FP += r['FP']; FN += r['FN']
    print(f"  {os.path.basename(gp)[:-5]}: J={r['jaccard']:.3f} matched {r['matched_gt']}/{r['n_gt_nodes']} pred={r['n_pred_nodes']}")
print(f'micro edge-Jaccard: {TP/(TP+FP+FN+1e-9):.4f}  (TP={TP} FP={FP} FN={FN})')

**Read the results:** val edge-Jaccard vs **v1's NN 0.808** (same linker), and `matched/n_gt` (detection
recall). This is a **bundle** (base24 + BatchNorm + residual + stronger aug) — if it beats v1, ABLATE via the
`NORM`/`RESIDUAL`/`BASE`/aug flags before spending an LB slot, so you know which change earned the gain.

**What changed vs `v1_unet_train.ipynb` (everything else identical):**
- `BASE` 16→24 (`chs = 24,48,96,192,384`); print the param count — expect ~13M (v1 = 5.8M). `BASE=32` if T4 fits.
- `NORM='batch'` (BatchNorm3d) vs v1 InstanceNorm. ⚠️ **BATCH=2 is small for BatchNorm** (noisy running stats);
  the reference used BN at a larger effective batch (pooled 64³). If BN val loss is unstable / worse than
  expected, flip `NORM='group'` (GroupNorm, small-batch-safe) or `'instance'` — one-line ablation.
- `RESIDUAL=True`: every ConvBlock is now `act(convs(x) + proj(x))`. `RESIDUAL=False`+`NORM='instance'` reproduces
  v1's block exactly.
- **Augmentation actually bites now:** v1 did `v *= uniform(0.9,1.1)` then percentile-normalized — the scale
  **cancels**, so v1 had only geometric aug. v5 adds in-plane 90° rotations + gamma/contrast/brightness/noise
  applied AFTER normalization. Aims at generalization to the unseen `44b6_` specimen — raise `EVAL_N_VAL` toward
  20 (both specimens) to actually see it (val is all `6bba_`).

**Memory:** base24 ≈ 2.25× v1's activations. Should fit a T4 at `BATCH=2`, `PATCH=(64,128,128)` with AMP. If OOM:
lower `BATCH` to 1, shrink `PATCH` xy to 96, or `BASE` back to 16. (xy patch must stay square for the rot90 aug.)

**Checkpoints / resume (its OWN files — do NOT attach v1/v4 weights):** every epoch writes `v5_UNet_res_best.pt`
(best weights) + `unet_res_latest.pt` (full: model+opt+scaler+scheduler+epoch+best+history) + `history.csv` to
`/kaggle/working`. To resume across sessions, save `/kaggle/working` as a dataset and add it as an input; the
train cell auto-finds `unet_res_latest.pt` and continues. Cosine LR / early-stop / time-budget safe-exit as v1.

**Eval-only:** set `EVAL_ONLY=True`, attach your `v5_UNet_res_best.pt` dataset, Run All — skips training and just
scores `EVAL_N_VAL` held-out val samples.

**Next:** local edge-J > 0.808 → better detector, worth an LB slot (submit these weights via a `submit.ipynb`
DETECTOR branch, HM_THR held at 0.3, single-variable vs 0.844). ≤ 0.808 → ablate or drop. If it wins, the arch
path (deeper / base32) stays open; if not, pivot GPU to Phase 2 learned linking (Trackastra).
